In [10]:
import pandas as pd
import numpy as np

# 1. 加载数据 (请将路径替换为你实际下载的文件路径)
print("Loading data...")
metadata = pd.read_csv('Model.csv')
crispr_data = pd.read_csv('CRISPRGeneEffect.csv', index_col=0) # 第一列通常是 ModelID，设为索引

print(f"CRISPR data shape: {crispr_data.shape}") 
# 预期输出类似: (1000+ 细胞系, 17000+ 基因)

Loading data...
CRISPR data shape: (1186, 18435)


In [11]:
# 2. 筛选 AML 细胞系
# 使用 Oncotree 标签进行精确匹配
aml_metadata = metadata[
    (metadata['OncotreeLineage'] == 'Myeloid') & 
    (metadata['OncotreePrimaryDisease'] == 'Acute Myeloid Leukemia')
]
aml_model_ids = aml_metadata['ModelID'].tolist()

# 3. 定义非 AML 细胞系 (作为背景对照)
all_model_ids = metadata['ModelID'].tolist()
non_aml_model_ids = [mid for mid in all_model_ids if mid not in aml_model_ids]

# 4. 从 CRISPR 矩阵中提取两组数据
# 确保 ID 都在 CRISPR 矩阵的索引中
aml_models_in_matrix = [mid for mid in aml_model_ids if mid in crispr_data.index]
non_aml_models_in_matrix = [mid for mid in non_aml_model_ids if mid in crispr_data.index]

aml_crispr = crispr_data.loc[aml_models_in_matrix]
non_aml_crispr = crispr_data.loc[non_aml_models_in_matrix]

print(f"Number of AML cell lines: {len(aml_models_in_matrix)}")
print(f"Number of Non-AML cell lines: {len(non_aml_models_in_matrix)}")

Number of AML cell lines: 30
Number of Non-AML cell lines: 1156


In [12]:
# 5. 计算每组中每个基因的 Median Score (使用中位数比平均数更稳健，不易受极端值影响)
aml_median_scores = aml_crispr.median(axis=0)
non_aml_median_scores = non_aml_crispr.median(axis=0)

# 将结果合并为一个 DataFrame 方便对比
dependency_df = pd.DataFrame({
    'AML_Median_Score': aml_median_scores,
    'Non_AML_Median_Score': non_aml_median_scores
})

# 计算特异性差值 (Delta)
# Delta 越小（越负），说明在 AML 中比在其他细胞系中更致命
dependency_df['Delta_Score'] = dependency_df['AML_Median_Score'] - dependency_df['Non_AML_Median_Score']

# 6. 设定阈值进行过滤
# 条件 A: 在 AML 中表现出强依赖性 (通常设定为 -0.5 或更严格的 -0.8)
aml_dependency_threshold = -0.3

# 条件 B: 剔除泛必需基因。要求在非 AML 中不太致命，或者 Delta 足够大。
# 这里我们采用双重过滤：既要求在 AML 中绝对值够低，又要求与背景有明显差异
delta_threshold = -0.1 # 意味着在 AML 中的分数比非 AML 背景至少低 0.3

final_aml_specific_genes = dependency_df[
    (dependency_df['AML_Median_Score'] < aml_dependency_threshold) & 
    (dependency_df['Delta_Score'] < delta_threshold)
]

# 按 Delta 分数排序，越靠前代表 AML 特异性依赖越强
final_aml_specific_genes = final_aml_specific_genes.sort_values(by='Delta_Score', ascending=True)

print(f"Found {len(final_aml_specific_genes)} AML-specific essential genes.")
print(final_aml_specific_genes.head(10)) # 查看排名前十的核心基因

Found 687 AML-specific essential genes.
                AML_Median_Score  Non_AML_Median_Score  Delta_Score
DCAF7 (10238)          -1.956244             -0.699600    -1.256644
CBFB (865)             -1.262616             -0.135454    -1.127162
MYB (4602)             -1.391211             -0.293012    -1.098199
CDK6 (1021)            -1.378271             -0.485266    -0.893005
NAMPT (10135)          -1.097090             -0.295665    -0.801425
EEF1A1 (1915)          -3.210430             -2.430703    -0.779728
WDR36 (134430)         -2.466974             -1.703577    -0.763397
SPI1 (6688)            -0.784639             -0.061158    -0.723481
UMPS (7372)            -1.133642             -0.417814    -0.715829
ADSL (158)             -1.626417             -0.918493    -0.707923


In [13]:
common_ess = pd.read_csv('AchillesCommonEssentialControls.csv',
                         header=None)
common_ess_names = [str(g).split(' ')[0].split('(')[0].strip()
                    for g in common_ess.iloc[:, 0].tolist()]
final_aml_specific_genes = final_aml_specific_genes[~final_aml_specific_genes.index.isin(common_ess_names)]

In [14]:
def clean_gene_name(col_name):
    # 将 "SPI1 (6688)" 变成 "SPI1"
    if isinstance(col_name, str) and '(' in col_name:
        return col_name.split(' (')[0]
    return col_name

final_aml_specific_genes.index = final_aml_specific_genes.index.map(clean_gene_name)

In [15]:
final_aml_specific_genes.to_csv("./DepMap_GT.csv")

In [16]:
gold_standard = pd.read_excel('/disk1/cai029/Caufussion/Benchmark_driver/Ground_truths/AML_Summary_C0023467.xlsx', engine='openpyxl')
driver_lst = gold_standard[gold_standard['score']>0.5]['gene_symbol'].tolist()
set(final_aml_specific_genes.index) & set(driver_lst)

/disk1/cai029/anaconda3/envs/celloracle_env/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{'CBFB',
 'CDK6',
 'CEBPA',
 'DDX41',
 'DHX15',
 'ERCC1',
 'KMT2A',
 'KRAS',
 'LYL1',
 'MYC',
 'PTPN11',
 'RUNX1',
 'SETD2',
 'SPI1',
 'TSC2',
 'TYMS'}

In [7]:
lst = ['STAT3','GATA2','STAT1','SPI1','RUNX1','RBBP5','TP53','ZNF143','GATA3',
       'ZBTB7A','ZEB1','STAT4','CHD7','ATF3','RXRA','JAK2','CEBPA','EOMES','PRDM1',
        ]
       
print(set(lst) & set(final_aml_specific_genes.index))

{'CEBPA', 'RUNX1', 'SPI1'}


In [9]:
print(set(lst) & set(driver_lst))

{'CEBPA', 'SPI1', 'ZBTB7A', 'GATA2', 'STAT3', 'TP53', 'JAK2', 'RUNX1'}


In [ ]:
"MYC" in set(driver_lst)